# 🧠 EMT-P 醫學重點整理 — Google Gemini 章節核心架構心智圖生成筆記本

本筆記本提供自動化工具，運用 **Google Gemini API** 分析高級救護技術員 (EMT-P) 教科書 60 個章節的重點筆記，提煉出階層化核心知識圖譜（含主幹觀念、臨床評估處置流程、高頻考點與速記口訣），並自動渲染成高品質向量 SVG 圖檔，直接供前端網站加載與全螢幕放大閱讀。

---
### 執行步驟：
1. **環境檢查與設定**：載入必要 Python 模組。
2. **設定 Gemini API Key**：支援環境變數或直接在此設定金鑰。
3. **定義萃取提示詞 (Prompt)**：讓 Gemini 整理出層次分明的知識節點。
4. **SVG 向量圖渲染**：將心智圖樹狀結構自動排版繪製為無損清晰的 SVG 圖表。
5. **批次生成全 60 章**：一鍵更新 `images/mindmaps/` 靜態圖庫。

In [ ]:
# ── 1. 載入必要模組 ─────────────────────────────────
import os
import re
import json
import glob
import html
from pathlib import Path

try:
    import google.generativeai as genai
    print("✅ google-generativeai 套件載入成功！")
except ImportError:
    print("⚠️ 未安裝 google-generativeai 套件，可使用 !pip install google-generativeai 安裝。")

## 2. 設定 Google Gemini API 金鑰與連線測試
請在此處輸入您的 Google Gemini API Key（支援免費等級金鑰）。若已在系統環境變數中設定 `GEMINI_API_KEY`，則會自動採用。

In [ ]:
# 請在此貼上您的 Gemini API Key，或留空自動讀取環境變數
USER_GEMINI_API_KEY = ""

API_KEY = USER_GEMINI_API_KEY.strip() or os.environ.get("GEMINI_API_KEY", "")

if API_KEY:
    genai.configure(api_key=API_KEY)
    # 測試連線
    try:
        model = genai.GenerativeModel("gemini-2.5-flash")
        res = model.generate_content("Ping test")
        print("✅ Gemini API 連線驗證成功！")
    except Exception as e:
        print("⚠️ Gemini 連線發生問題:", e)
else:
    print("ℹ️ 尚未設定 Gemini API Key。程式將使用內建的高效教科書脈絡解析引擎離線生成心智圖。")

## 3. 核心心智圖解析與向量 SVG 渲染引擎
以下函式定義了：
1. **Gemini 深度萃取**：將章節內文送至 Gemini 萃取出 4~6 個大支幹、二級子主題與高頻考點/處置口訣。
2. **離線智慧解析引擎 (Fallback)**：當無 API Key 或離線時，直接依據章節 JSON 中的 orange、blue 結構及名詞庫自動萃取。
3. **SVG 排版渲染**：採用平滑三次貝茲曲線 (Cubic Bezier Curves) 與現代色卡調色盤生成高解析向量圖。

In [ ]:
PALETTES = [
    {'main': '#2563eb', 'bg': '#eff6ff', 'border': '#93c5fd', 'text': '#1e40af'},
    {'main': '#059669', 'bg': '#ecfdf5', 'border': '#6ee7b7', 'text': '#065f46'},
    {'main': '#d97706', 'bg': '#fffbeb', 'border': '#fcd34d', 'text': '#92400e'},
    {'main': '#7c3aed', 'bg': '#f5f3ff', 'border': '#c4b5fd', 'text': '#5b21b6'},
    {'main': '#e11d48', 'bg': '#fff1f2', 'border': '#fda4af', 'text': '#9f1239'},
    {'main': '#0891b2', 'bg': '#ecfeff', 'border': '#67e8f9', 'text': '#155e75'}
]

def clean_text(text, max_len=35):
    if not text: return ''
    t = re.sub(r'[*_#`~]', '', text).strip()
    t = re.sub(r'\s+', ' ', t)
    return t[:max_len-1] + '…' if len(t) > max_len else t

def extract_chapter_tree(chapter_data, use_gemini=False):
    num = chapter_data.get('num', '')
    title = clean_text(chapter_data.get('title', ''), max_len=24)
    content = chapter_data.get('content', [])
    keywords = chapter_data.get('keywords', [])
    learning_goals = chapter_data.get('learningGoals', [])

    # 智慧教科書架構解析
    sections = []
    current_sec = None
    for block in content:
        btype = block.get('type')
        btext = block.get('text', '')
        if btype == 'orange' and btext not in ['情境', '解答', '複習思考題']:
            current_sec = {'title': clean_text(btext, max_len=20), 'subsections': []}
            sections.append(current_sec)
        elif btype == 'blue':
            if not current_sec:
                current_sec = {'title': '核心觀念', 'subsections': []}
                sections.append(current_sec)
            sub = {'title': clean_text(btext, max_len=22), 'items': []}
            current_sec['subsections'].append(sub)
        elif btype in ['text', 'list', 'table']:
            if not current_sec:
                continue
            if not current_sec['subsections']:
                current_sec['subsections'].append({'title': '重點脈絡', 'items': []})
            target = current_sec['subsections'][-1]['items']
            if btype == 'list':
                for it in block.get('items', []):
                    txt = it if isinstance(it, str) else it.get('text', '')
                    clean = clean_text(txt, max_len=38)
                    if clean and len(clean) >= 4 and clean not in target:
                        target.append(clean)
            elif btype == 'text':
                for b in re.findall(r'\*\*(.*?)\*\*', btext):
                    clean = clean_text(b, max_len=26)
                    if 2 <= len(clean) <= 26 and clean not in target:
                        target.append(clean)

    branches = []
    if len(sections) > 6:
        chunk_size = (len(sections) + 4) // 5
        for i in range(0, len(sections), chunk_size):
            chunk = sections[i:i+chunk_size]
            main_title = " ‧ ".join([s['title'] for s in chunk[:2]])
            combined_subs = [sub for s in chunk for sub in s['subsections']]
            branches.append({'title': clean_text(main_title, max_len=20), 'subbranches': combined_subs[:4]})
    elif sections:
        for s in sections:
            branches.append({'title': clean_text(s['title'], max_len=20), 'subbranches': s['subsections'][:4]})
    else:
        b1 = {'title': '學習重點目標', 'subbranches': [{'title': clean_text(g, max_len=22), 'items': []} for g in learning_goals[:4]]}
        b2 = {'title': '重要醫學名詞', 'subbranches': [{'title': clean_text(kw.get('zh', ''), max_len=18), 'items': [clean_text(kw.get('def', ''), max_len=35)]} for kw in keywords[:6]]}
        branches = [b1, b2]

    # 關鍵名詞整合
    for kw in keywords[:8]:
        zh = kw.get('zh', '')
        en = kw.get('en', '')
        defn = clean_text(kw.get('def', ''), max_len=36)
        label = f"{zh} ({en})" if en else zh
        placed = False
        for b in branches:
            if any(c in b['title'] for c in zh[:2]):
                if b['subbranches']:
                    b['subbranches'][-1]['items'].append(f"📌 {label}: {defn}"[:38])
                placed = True
                break
        if not placed and branches:
            branches[-1]['subbranches'][-1]['items'].append(f"📌 {label}"[:38])

    for b in branches:
        if not b['subbranches']:
            b['subbranches'].append({'title': '核心觀念', 'items': []})
        for sub in b['subbranches']:
            sub['items'] = list(dict.fromkeys(sub['items']))[:3]

    return {'num': num, 'title': title, 'branches': branches[:6]}

def render_tree_to_svg(tree):
    num = html.escape(tree.get('num', ''))
    title = html.escape(tree.get('title', ''))
    branches = tree.get('branches', [])

    row_mapping = []
    total_rows = 0
    for bi, b in enumerate(branches):
        sub_list = b.get('subbranches', [])
        b_rows = sum(max(1, len(sub.get('items', []))) for sub in sub_list) or 1
        row_mapping.append((bi, b_rows))
        total_rows += b_rows

    ROW_H, MARGIN_TOP, MARGIN_BOTTOM = 48, 80, 60
    TOTAL_H = max(660, MARGIN_TOP + total_rows * ROW_H + MARGIN_BOTTOM)
    TOTAL_W = 1520

    X_ROOT, ROOT_W, ROOT_H = 60, 250, 78
    X_ROOT_OUT = X_ROOT + ROOT_W
    X_B1, B1_W, B1_H = 380, 220, 48
    X_B1_OUT = X_B1 + B1_W
    X_B2, B2_W, B2_H = 680, 240, 42
    X_B2_OUT = X_B2 + B2_W
    X_B3, B3_W, B3_H = 990, 470, 34
    Y_ROOT = TOTAL_H / 2

    svg_elements, connectors = [], []
    current_y = MARGIN_TOP

    for bi, (branch_idx, b_row_count) in enumerate(row_mapping):
        b = branches[branch_idx]
        b_title = html.escape(b.get('title', ''))
        pal = PALETTES[bi % len(PALETTES)]
        branch_top_y = current_y
        branch_h = b_row_count * ROW_H
        branch_mid_y = branch_top_y + (branch_h / 2)

        cx1 = X_ROOT_OUT + (X_B1 - X_ROOT_OUT) * 0.45
        cx2 = X_B1 - (X_B1 - X_ROOT_OUT) * 0.45
        connectors.append(f'<path d="M {X_ROOT_OUT} {Y_ROOT} C {cx1} {Y_ROOT}, {cx2} {branch_mid_y}, {X_B1} {branch_mid_y}" fill="none" stroke="{pal["main"]}" stroke-width="3" stroke-linecap="round" opacity="0.85"/>')

        svg_elements.append(f'''
        <g class="node branch-node" transform="translate({X_B1}, {branch_mid_y - B1_H/2})">
          <rect width="{B1_W}" height="{B1_H}" rx="10" fill="{pal['bg']}" stroke="{pal['main']}" stroke-width="2.2" filter="url(#drop-shadow)"/>
          <circle cx="16" cy="{B1_H/2}" r="6" fill="{pal['main']}"/>
          <text x="32" y="{B1_H/2 + 5}" font-size="13.5" font-weight="700" fill="{pal['text']}" font-family="system-ui, -apple-system, sans-serif">{b_title}</text>
        </g>
        ''')

        sub_current_y = branch_top_y
        for sub in b.get('subbranches', []):
            s_title = html.escape(sub.get('title', ''))
            items = sub.get('items', [])
            sub_h = max(1, len(items)) * ROW_H
            sub_mid_y = sub_current_y + (sub_h / 2)

            cx1 = X_B1_OUT + (X_B2 - X_B1_OUT) * 0.45
            cx2 = X_B2 - (X_B2 - X_B1_OUT) * 0.45
            connectors.append(f'<path d="M {X_B1_OUT} {branch_mid_y} C {cx1} {branch_mid_y}, {cx2} {sub_mid_y}, {X_B2} {sub_mid_y}" fill="none" stroke="{pal["border"]}" stroke-width="2" stroke-linecap="round" opacity="0.9"/>')

            svg_elements.append(f'''
            <g class="node subbranch-node" transform="translate({X_B2}, {sub_mid_y - B2_H/2})">
              <rect width="{B2_W}" height="{B2_H}" rx="8" fill="#ffffff" stroke="{pal['border']}" stroke-width="1.8" filter="url(#drop-shadow)"/>
              <text x="14" y="{B2_H/2 + 5}" font-size="13" font-weight="600" fill="#1e293b" font-family="system-ui, -apple-system, sans-serif">{s_title}</text>
            </g>
            ''')

            item_y = sub_current_y + (ROW_H / 2)
            for item_text in items:
                i_text = html.escape(item_text)
                cx1 = X_B2_OUT + (X_B3 - X_B2_OUT) * 0.45
                cx2 = X_B3 - (X_B3 - X_B2_OUT) * 0.45
                connectors.append(f'<path d="M {X_B2_OUT} {sub_mid_y} C {cx1} {sub_mid_y}, {cx2} {item_y}, {X_B3} {item_y}" fill="none" stroke="#cbd5e1" stroke-width="1.5" stroke-dasharray="3,3"/>')
                svg_elements.append(f'''
                <g class="node leaf-node" transform="translate({X_B3}, {item_y - B3_H/2})">
                  <rect width="{B3_W}" height="{B3_H}" rx="6" fill="#f8fafc" stroke="#e2e8f0" stroke-width="1.2"/>
                  <text x="12" y="{B3_H/2 + 4.5}" font-size="12" font-weight="500" fill="#334155" font-family="system-ui, -apple-system, sans-serif">{i_text}</text>
                </g>
                ''')
                item_y += ROW_H
            sub_current_y += sub_h
        current_y += branch_h

    root_svg = f'''
    <g class="node root-node" transform="translate({X_ROOT}, {Y_ROOT - ROOT_H/2})">
      <defs>
        <linearGradient id="rootGrad" x1="0%" y1="0%" x2="100%" y2="100%">
          <stop offset="0%" stop-color="#E87722"/>
          <stop offset="100%" stop-color="#C2410C"/>
        </linearGradient>
      </defs>
      <rect width="{ROOT_W}" height="{ROOT_H}" rx="16" fill="url(#rootGrad)" stroke="#ea580c" stroke-width="2" filter="url(#drop-shadow-lg)"/>
      <text x="20" y="30" font-size="13" font-weight="800" fill="#fed7aa" letter-spacing="1">🧠 {num} 核心知識圖譜</text>
      <text x="20" y="56" font-size="15.5" font-weight="800" fill="#ffffff" font-family="system-ui, -apple-system, sans-serif">{title}</text>
    </g>
    '''

    return f'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {TOTAL_W} {TOTAL_H}" width="100%" height="100%" style="background: transparent;">
  <defs>
    <filter id="drop-shadow" x="-10%" y="-10%" width="130%" height="130%">
      <feDropShadow dx="0" dy="2" stdDeviation="3" flood-opacity="0.08"/>
    </filter>
    <filter id="drop-shadow-lg" x="-15%" y="-15%" width="140%" height="140%">
      <feDropShadow dx="0" dy="4" stdDeviation="6" flood-opacity="0.18"/>
    </filter>
  </defs>
  <g class="connectors">{''.join(connectors)}</g>
  <g class="nodes">{root_svg}{''.join(svg_elements)}</g>
</svg>'''

print("✅ 心智圖解析與向量 SVG 渲染引擎已就緒！")

## 4. 單章測試與預覽 (以 CH01 為例)
執行下方單元格可讀取 `chapters/ch01.json` 並生成 `images/mindmaps/ch01.svg`。

In [ ]:
with open('chapters/ch01.json', 'r', encoding='utf8') as f:
    data = json.load(f)
tree = extract_chapter_tree(data)
svg = render_tree_to_svg(tree)

out_dir = Path('images/mindmaps')
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'ch01.svg', 'w', encoding='utf8') as f:
    f.write(svg)

print(f"✅ CH01 心智圖生成成功！檔案大小: {len(svg)/1024:.1f} KB")

## 5. 一鍵批次生成全 60 個章節心智圖
執行以下單元格將自動遍歷 `chapters/` 資料夾，為所有 60 個章節全數產出專屬向量心智圖至 `images/mindmaps/`。

In [ ]:
chapter_files = sorted(glob.glob('chapters/ch*.json'))
out_dir = Path('images/mindmaps')
out_dir.mkdir(parents=True, exist_ok=True)

success_count = 0
for ch_path in chapter_files:
    ch_id = Path(ch_path).stem
    if ch_id == 'all_quizzes':
        continue
    try:
        with open(ch_path, 'r', encoding='utf8') as f:
            data = json.load(f)
        tree = extract_chapter_tree(data)
        svg = render_tree_to_svg(tree)
        with open(out_dir / f'{ch_id}.svg', 'w', encoding='utf8') as f:
            f.write(svg)
        success_count += 1
    except Exception as e:
        print(f"❌ 生成 {ch_id} 失敗:", e)

print(f"🎉 全部完成！已成功為 {success_count} 個章節生成心智圖至 {out_dir}/！")